In [1]:
import os
import pandas as pd

# Chargement robuste du fichier nettoyé
chemin = (
    "data/titanic_propre.csv"
    if os.path.exists("data/titanic_propre.csv")
    else "../data/titanic_propre.csv"
)
df = pd.read_csv(chemin)

print(f"Jeu de données chargé : {df.shape[0]} passagers")

# 1. Les femmes ont-elles mieux survécu que les hommes dans chaque classe ?
survie_classe_sexe = (
    pd.pivot_table(df, values="survived", index="class", columns="sex", aggfunc="mean")
    * 100
).round(1)
print("\n--- 1. Taux de survie par classe et sexe (%) ---")
print(survie_classe_sexe)

# 2. Les enfants ont-ils été prioritaires ? Taux de survie par tranche d'âge
df["tranche_age"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 60, 100],
    labels=["enfant", "ado", "adulte", "senior"],
)
survie_age = df.groupby("tranche_age", observed=False)["survived"].agg(
    effectif="count",
    taux_survie=lambda x: round(x.mean() * 100, 1),
)
print("\n--- 2. Taux de survie par tranche d'âge ---")
print(survie_age)

# 3. Port au prix moyen le plus élevé et répartition des classes
prix_moyen_port = df.groupby("embark_town")["fare"].mean().round(2)
repartition_classes = (
    pd.crosstab(df["embark_town"], df["class"], normalize="index") * 100
).round(1)
print("\n--- 3. Prix moyen par port d'embarquement ($) ---")
print(prix_moyen_port)
print("\nRépartition des classes par port (%) :")
print(repartition_classes)

# 4. Voyager seul augmentait-il ou diminuait-il les chances de survie ?
survie_solitude = df.groupby("alone")["survived"].agg(
    effectif="count",
    taux_survie=lambda x: round(x.mean() * 100, 1),
)
print("\n--- 4. Survie selon la situation de voyage (alone) ---")
print(survie_solitude)


Jeu de données chargé : 889 passagers

--- 1. Taux de survie par classe et sexe (%) ---
sex     female  male
class               
First     96.7  36.9
Second    92.1  15.7
Third     50.0  13.5

--- 2. Taux de survie par tranche d'âge ---
             effectif  taux_survie
tranche_age                       
enfant             69         58.0
ado                70         42.9
adulte            729         36.5
senior             21         19.0

--- 3. Prix moyen par port d'embarquement ($) ---
embark_town
Cherbourg      59.95
Queenstown     13.28
Southampton    27.08
Name: fare, dtype: float64

Répartition des classes par port (%) :
class        First  Second  Third
embark_town                      
Cherbourg     50.6    10.1   39.3
Queenstown     2.6     3.9   93.5
Southampton   19.7    25.5   54.8

--- 4. Survie selon la situation de voyage (alone) ---
       effectif  taux_survie
alone                       
False       354         50.6
True        535         30.1


### Réponses aux questions d'analyse métier

1. **Survie femmes vs hommes** : **Oui**, les femmes ont nettement mieux survécu que les hommes dans toutes les classes (1re : **96.7 %** vs 36.9 %, 2e : **92.1 %** vs 15.7 %, 3e : **50.0 %** vs 13.5 %).
2. **Priorité aux enfants** : **Oui**, les enfants (0-12 ans) ont été prioritaires avec le taux de survie le plus élevé (**58.0 %**), soit 3 fois plus que les seniors (**19.0 %**).
3. **Prix par port et classes** : **Cherbourg** a le prix moyen le plus élevé (**59.95 $**, contre 27.08 $ à Southampton et 13.28 $ à Queenstown). Cela s'explique directement par la composition sociale : plus de la moitié des passagers de Cherbourg (**50.6 %**) voyageaient en 1re classe.
4. **Voyager seul** : Voyager seul **diminuait fortement** les chances de survie (**30.1 %** de survie pour les passagers seuls contre **50.6 %** pour les passagers accompagnés d'au moins un proche).
